In [ ]:
# Kurulum: Colab'da repoyu indirir, yerelde hiçbir şey yapmaz.
import os, sys
REPO = "https://github.com/mehmetalibeylikci/python-cevresel-modelleme"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/python-cevresel-modelleme"):
        !git clone -q {REPO} /content/python-cevresel-modelleme
    %cd /content/python-cevresel-modelleme
    %pip install -q sweetviz
else:
    while not os.path.exists("src/model.py") and os.getcwd() != "/":
        os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
# Bu modülün ek paketleri. Colab'da kurulur; yerelde: pip install -r bonus/fizik-mi-veri-mi/requirements.txt
if "google.colab" in sys.modules:
    %pip install -q -r bonus/fizik-mi-veri-mi/requirements.txt

# Bonus — Fizik mi, Veri mi?

**İYTE Çevre Topluluğu · Python ile Çevresel Modelleme · Kendi başına çalışma (~1,5 saat)**

Üç gün boyunca bir **fiziksel model** kurduk: kütle korunumunu bir denkleme çevirdik, her terimin anlamını biliyorduk. Bugünlerde pek çok problem ise **makine öğrenmesi (ML)** ile çözülüyor: denklemi biz yazmıyoruz, model veriden kendisi öğreniyor.

Bu modülde aynı soruyu ML ile soracağız ve sonra **SHAP** ile ML modelinin "neye baktığını" açacağız.

**Ön koşul:** Gün 1 ve Gün 2.

> ⚠️ **Önemli bir dürüstlük notu:** Tahtalı'nın geçmiş doluluk serisi açık bir kaynakta yok. Bu yüzden ML modelini **gözlemlere değil, kendi fiziksel modelimizin çıktısına** eğiteceğiz. Yani ML modeli gerçeği değil, bizim kova modelimizi öğrenmeye çalışacak. Bu kurgu yine de çok öğretici: ML modelinin, bizim elle yazdığımız fiziği veriden **yeniden keşfedip keşfedemeyeceğini** görebiliriz. Gerçek bir çalışmada hedef değişken mutlaka gözlem olmalıdır.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.model import S0_DATE, S0_HM3, simulate

monthly = pd.read_csv("data/processed/tahtali_monthly.csv")
monthly.index = pd.to_datetime(monthly.pop("month"))
data = monthly.loc[S0_DATE:].copy()

physics = simulate(data["precipitation_mm"], data["et0_mm"], S0_HM3)

## 1. Hedef ve özellikler

- **Hedef (y):** Bir aydaki depolama değişimi, $\Delta S$ (hm³). Barajın o ay ne kadar dolduğu ya da boşaldığı.
- **Özellikler (X):** O ayın yağışı, ET0'ı ve takvim ayı (1–12).

Fiziksel modelimizde $\Delta S$ bu üç bilgiyle **neredeyse** tamamen belirleniyor. "Neredeyse" diyoruz, çünkü baraj doluyken taşma, boşken ölü hacim sınırı devreye giriyor ve bu durum depolamanın kendisine bağlı. ML modeli depolamayı görmüyor. Bunun sonuca etkisini birazdan göreceğiz.

In [ ]:
previous = physics["storage"].shift(1)
previous.iloc[0] = S0_HM3
data["delta_storage"] = physics["storage"] - previous
data["month_of_year"] = data.index.month

features = ["precipitation_mm", "et0_mm", "month_of_year"]
X, y = data[features], data["delta_storage"]
data.head()

## 2. Eğitim ve test verisini **zamana göre** ayırmak

ML'de modeli bir kısım veriyle eğitip, **hiç görmediği** başka bir kısımla test ederiz. Zaman serilerinde bu ayrım **rastgele yapılmaz**: rastgele ayırırsak model, test ayının bir ay öncesini ve sonrasını eğitimde görmüş olur. Bu, geleceği bilerek geçmişi tahmin etmek gibidir (*bilgi sızıntısı*). Doğrusu: geçmişle eğit, gelecekle test et.

In [ ]:
split = "2024-01-01"
X_train, X_test = X.loc[:split].iloc[:-1], X.loc[split:]
y_train, y_test = y.loc[:split].iloc[:-1], y.loc[split:]
print(f"Eğitim: {len(X_train)} ay ({X_train.index[0]:%Y-%m} → {X_train.index[-1]:%Y-%m})")
print(f"Test:   {len(X_test)} ay ({X_test.index[0]:%Y-%m} → {X_test.index[-1]:%Y-%m})")

## 3. Random Forest

**Random Forest**, çok sayıda karar ağacının ortalamasını alan, güçlü ve kolay kullanılan bir ML modelidir. scikit-learn'de her model aynı üç adımla çalışır: oluştur → `fit` → `predict`.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

model = RandomForestRegressor(n_estimators=300, random_state=42)
model.fit(X_train, y_train)
prediction = pd.Series(model.predict(X_test), index=X_test.index)

print(f"R²   = {r2_score(y_test, prediction):.2f}")
print(f"RMSE = {np.sqrt(mean_squared_error(y_test, prediction)):.2f} hm³/ay")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(y.index, y, color="black", label="Fiziksel model (hedef)")
ax.plot(prediction.index, prediction, color="tab:green", marker="o", markersize=3, label="Random Forest tahmini")
ax.axvline(pd.Timestamp(split), color="gray", linestyle="--")
ax.text(pd.Timestamp(split), ax.get_ylim()[1] * 0.9, "  test dönemi →", color="gray")
ax.axhline(0, color="black", linewidth=0.5)
ax.set_ylabel("ΔS (hm³/ay)")
ax.set_title("Aylık depolama değişimi: fiziksel model ve Random Forest")
ax.legend(loc="lower left")
plt.show()

### 🟢 Alıştırma 1
Random Forest hangi aylarda en çok yanılıyor? Test dönemindeki en büyük 3 hatayı bulun ve o aylarda fiziksel modelde taşma (`physics["spill"]`) olup olmadığına bakın.

In [ ]:
# TODO: kendin yaz
...

Büyük hatalar genellikle **taşma** olan aylarda. Baraj doluyken yağan yağmur depolamayı artırmaz, taşar. Ama ML modeli barajın dolu olup olmadığını bilmiyor, çünkü depolama özelliklerin arasında yok. Fiziksel model ise bunu yapısı gereği biliyor.

## 4. SHAP: Model neye bakıyor?

ML modelleri çoğu zaman "kara kutu" olarak eleştirilir. **SHAP** (SHapley Additive exPlanations), her tahmini özelliklerin katkılarına ayırır: "Bu ay ΔS'nin tahmini +5 hm³; bunun +7'si yağıştan, −2'si ET0'dan geliyor" gibi.

In [ ]:
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer(X_test)

shap.plots.beeswarm(shap_values, show=False)
plt.title("Her özelliğin tahmine katkısı (test dönemi)")
plt.show()

**Nasıl okunur?**
- Her nokta bir ay. Yatay konum, o özelliğin o ayın tahminine katkısı (hm³).
- Renk, özelliğin değeri: kırmızı yüksek, mavi düşük.
- Özellikler, ortalama etkilerine göre yukarıdan aşağı sıralanır.

Beklentimiz: yağış en önemli özellik olmalı, ve yüksek yağış (kırmızı) pozitif katkı vermeli.

In [ ]:
shap.plots.scatter(shap_values[:, "precipitation_mm"], show=False)
plt.title("Yağışın katkısı yağışa nasıl bağlı?")
plt.show()

Fiziksel modelde yağışın etkisi **doğrusal**dı: $(C \cdot A_{havza} + A_{göl}) \cdot P \approx 0{,}2$ hm³/mm. SHAP grafiğinde ilişki yaklaşık doğrusal mı? Yüksek yağışlarda neden düzleşiyor olabilir? (İpucu: taşma.)

### 🔵 Alıştırma 2
Fiziksel modeldeki yağış katsayısını, $0{,}33 \cdot 546 \cdot 0{,}001 + 23{,}52 \cdot 0{,}001$ hm³/mm, hesaplayın. Sonra SHAP değerlerine bir doğru uydurarak (`np.polyfit`) ML modelinin "öğrendiği" eğimi bulun ve karşılaştırın.

In [ ]:
# TODO: kendin yaz
...

## 5. Tartışma

1. **ML modeli fiziği yeniden keşfedebilir mi?** Yağışın etkisinin yönünü ve büyüklüğünü kabaca buldu. Taşma ve ölü hacim gibi **eşik davranışlarını** ise, ilgili bilgi (depolama) verilmeden öğrenemedi.
2. **Gerçek gözlem olsaydı ne değişirdi?** Hedef, kendi modelimizin çıktısı yerine İZSU'nun ölçtüğü değişim olsaydı, ML modeli bizim modelimizin **bilmediği** süreçleri (sızma, değişken çekim) de öğrenebilirdi. Ama bunun için yıllarca düzenli gözlem gerekir. Bizim elimizde yalnızca 5 gözlem vardı.
3. **Hangisi daha iyi?** Az veri varken ve süreçleri biliyorken fiziksel modeller; çok veri varken ve süreçler karmaşıkken ML öne çıkar. Pek çok modern çalışma ikisini birleştirir (*hibrit modeller*).

### AutoML hakkında bir not
**PyCaret** gibi AutoML araçları, tek satırla onlarca ML modelini deneyip en iyisini seçer. Hızlı bir ilk bakış için kullanışlıdır. Ama bu eğitimin mesajıyla dikkatli düşünülmeli: hangi modelin neden seçildiğini, hangi varsayımlarla çalıştığını ve verinin nasıl ayrıldığını anlamadan alınan "en iyi skor", yanıltıcı olabilir. Önce bu modüldeki gibi tek bir modeli elle kurup anlamanızı öneririz.

---
## Zaman kalırsa

### 🔵 Alıştırma 3
Özelliklere bir önceki ayın sonundaki **depolamayı** (`previous`) ekleyin ve modeli yeniden eğitin. Test hatası nasıl değişiyor? SHAP grafiğinde depolama nasıl bir rol oynuyor?

In [ ]:
# TODO: kendin yaz
...